In [1]:
import pandas as pd

df = pd.read_csv("data/resale-prices.csv")
df.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


In [2]:
import pandas as pd

df = pd.read_csv("data/resale-prices.csv")

# year
df["year"] = df["month"].str[:4].astype(int)

# lease years left
df["lease_years"] = df["remaining_lease"].str[:2].astype(int)

# price per sqm
df["price_per_sqm"] = df["resale_price"] / df["floor_area_sqm"]

df[["month", "year", "remaining_lease", "lease_years", "price_per_sqm"]].head()

,month,year,remaining_lease,lease_years,price_per_sqm
0,2017-01,2017,61 years 04 months,61,5272.727273
1,2017-01,2017,60 years 07 months,60,3731.343284
2,2017-01,2017,62 years 05 months,62,3910.447761
3,2017-01,2017,62 years 01 month,62,3897.058824
4,2017-01,2017,62 years 05 months,62,3955.223881


In [3]:
# median price per sqm by town, 2021 vs 2025
q1 = df[df["year"].isin([2021, 2025])]
q1 = q1.groupby(["town", "year"])["price_per_sqm"].median().unstack()

q1["change_%"] = ((q1[2025] - q1[2021]) / q1[2021] * 100).round(1)

q1.sort_values("change_%", ascending=False).head(10)

year,2021,2025,change_%
town,,,
SEMBAWANG,4408.602151,6796.460177,54.2
TOA PAYOH,5172.082604,7826.086957,51.3
BUKIT BATOK,4552.786070,6451.612903,41.7
TAMPINES,4784.810127,6682.692308,39.7
SENGKANG,4967.741935,6903.225806,39.0
HOUGANG,4750.000000,6582.089552,38.6
SERANGOON,5166.666667,7080.729167,37.0
PUNGGOL,5403.024452,7391.304348,36.8
BUKIT TIMAH,6485.398860,8866.100464,36.7


### Q1 answer

- Sembawang went up the most (+54%), then Toa Payoh (+51%)
- Punggol went up +37%
- Cheaper towns like Sembawang and Woodlands grew a lot, so cheaper areas are catching up

In [4]:
# 2025 only, grouped by lease left
q2 = df[df["year"] == 2025].copy()

bins = [0, 60, 70, 80, 90, 100]
labels = ["under 60", "60-70", "70-80", "80-90", "90+"]
q2["lease_group"] = pd.cut(q2["lease_years"], bins=bins, labels=labels)

q2.groupby("lease_group", observed=True)["price_per_sqm"].agg(["median", "count"]).round(0)

,median,count
lease_group,,
under 60,6119.0,7673
60-70,5945.0,4166
70-80,6045.0,4778
80-90,7660.0,3900
90+,7742.0,4567


### Q2 answer

- Flats with 80+ years left cost about 28% more per sqm than flats with 60-80 years left
- Below 80 years, prices stay around the same (about $6,000 per sqm)
- Flats under 60 years cost slightly more than 60-70, likely because many are in mature estates

In [5]:
# punggol vs all of singapore, each year
sg = df.groupby("year")["price_per_sqm"].median()
punggol = df[df["town"] == "PUNGGOL"].groupby("year")["price_per_sqm"].median()

q3 = pd.DataFrame({"singapore": sg, "punggol": punggol}).round(0)
q3["punggol_vs_sg_%"] = ((q3["punggol"] - q3["singapore"]) / q3["singapore"] * 100).round(1)

q3

,singapore,punggol,punggol_vs_sg_%
year,,,
2017,4282.0,4599.0,7.4
2018,4209.0,4677.0,11.1
2019,4176.0,4940.0,18.3
2020,4356.0,5108.0,17.3
2021,4916.0,5403.0,9.9
2022,5373.0,5965.0,11.0
2023,5709.0,6452.0,13.0
2024,6098.0,6935.0,13.7
2025,6500.0,7391.0,13.7


In [6]:
# lease groups for the whole dataset (used in tableau)
df["lease_group"] = pd.cut(df["lease_years"], bins=bins, labels=labels)

# keep only the columns we need
clean = df[["month", "year", "town", "flat_type", "floor_area_sqm",
            "lease_years", "lease_group", "resale_price", "price_per_sqm"]]

clean.to_csv("data/hdb_clean.csv", index=False)
print("saved", len(clean), "rows")

saved 245351 rows
